## **1. Import các thư viện cần thiết**

In [ ]:
import string
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from typing import List

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## **2. Đọc bộ dữ liệu**

In [54]:
DATASET_PATH = '../dataset/data.csv'
df = pd.read_csv(DATASET_PATH)
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [55]:
messages = df['Message'].values.tolist()
labels = df['Category'].values.tolist()

## **3. Tiền xử lý dữ liệu**

In [57]:
STOP_WORDS = set(nltk.corpus.stopwords.words("english"))
STEMMER = nltk.PorterStemmer()


class TextPreprocessor:
    """Text preprocessing pipeline"""

    def lowercase(self, text: str) -> str:
        """Convert text to lowercase."""
        return text.lower()

    def punctuation_removal(self, text: str) -> str:
        """Remove punctuation from text."""
        return text.translate(str.maketrans("", "", string.punctuation))

    def tokenize(self, text: str) -> list:
        """Tokenize text into words."""
        return nltk.word_tokenize(text)

    def remove_stop_words(self, tokens: List[str]) -> List[str]:
        """Remove stop words from token list."""
        return [token for token in tokens if token not in STOP_WORDS]

    def stemming(self, tokens: List[str]) -> List[str]:
        """Apply stemming to tokens."""
        return [STEMMER.stem(token) for token in tokens]

    def preprocess(self, text: str) -> List[str]:
        """Run the full preprocessing pipeline on the input text."""
        text = self.lowercase(text)
        text = self.punctuation_removal(text)
        tokens = self.tokenize(text)
        tokens = self.remove_stop_words(tokens)
        tokens = self.stemming(tokens)
        return tokens


## **4. Bag of words**

In [59]:
class BagOfWordsVectorizer:
    """Convert preprocessed tokens into a bag-of-words vector."""

    def __init__(self):
        self.dictionary = []

    def fit(self, messages: List[List[str]]):
        """Build the dictionary from the training messages."""
        for tokens in messages:
            for token in tokens:
                if token not in self.dictionary:
                    self.dictionary.append(token)

    def transform(self, tokens: List[str]) -> np.ndarray:
        """Convert messages into bag-of-words vectors."""
        features = np.zeros(len(self.dictionary))

        for token in tokens:
            if token in self.dictionary:
                features[self.dictionary.index(token)] += 1

        return features

    def fit_transform(self, messages: List[List[str]]) -> np.ndarray:
        """Fit the vectorizer and transform the messages."""
        self.fit(messages)
        return np.array([self.transform(message) for message in messages])


### **5. Models**

### **5.1 Naive Bayes Classifier**

In [69]:
class NaiveBayesClassifier:
    """A Naive Bayes model for sentiment analysis."""

    def __init__(self):
        self.model = MultinomialNB()

    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)

    def predict(self, X_test):
        return self.model.predict(X_test)

    def save_model(self, file_path):
        joblib.dump(self.model, file_path)

    def load_model(self, file_path):
        self.model = joblib.load(file_path)


### **5.2 K-Nearest Neighbor**

In [70]:
class KNNClassifier:
    def __init__(self, n_neighbors=5):
        self.model = KNeighborsClassifier(n_neighbors=n_neighbors)

    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)

    def predict(self, X_test):
        return self.model.predict(X_test)

    def save_model(self, file_path):
        joblib.dump(self.model, file_path)

    def load_model(self, file_path):
        self.model = joblib.load(file_path)


### **5.3 Logistic Regression**

In [71]:
class LogisticRegressionClassifier:
    def __init__(self):
        self.model = LogisticRegression(max_iter=1000)

    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)

    def predict(self, X_test):
        return self.model.predict(X_test)

    def save_model(self, file_path):
        joblib.dump(self.model, file_path)

    def load_model(self, file_path):
        self.model = joblib.load(file_path)

### **5.4 TF-IDF Vectorization**

In [72]:
class TFIDFVectorizer:
    def __init__(self, max_features=5000):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2)   # rất hợp spam detection
        )

    def fit_transform(self, texts):
        return self.vectorizer.fit_transform(texts)

    def transform(self, texts):
        return self.vectorizer.transform(texts)

    def save(self, file_path):
        joblib.dump(self.vectorizer, file_path)

    def load(self, file_path):
        self.vectorizer = joblib.load(file_path)

### **5.5. Support Vector Machine**

In [73]:
class SVMClassifier:
    def __init__(self):
        self.model = LinearSVC()

    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)

    def predict(self, X_test):
        return self.model.predict(X_test)

    def save_model(self, file_path):
        joblib.dump(self.model, file_path)

    def load_model(self, file_path):
        self.model = joblib.load(file_path)

## **6. Trainer**

In [ ]:
VAL_SIZE = 0.2
TEST_SIZE = 0.125
SEED = 0


class Trainer:
    """Trainer class to orchestrate the training pipeline."""

    def __init__(self):
        self.preprocessor = TextPreprocessor()
        self.vectorizer = BagOfWordsVectorizer()
        self.label_encoder = LabelEncoder()
        self.model = NaiveBayesClassifier()

    def train(self, messages, labels):
        """Train the model on the provided texts and labels."""
        # Preprocess the texts
        messages = [self.preprocessor.preprocess(message) for message in messages]

        # Vectorize the preprocessed texts
        X = self.vectorizer.fit_transform(messages)

        # Encode labels
        y = self.label_encoder.fit_transform(labels)

        # split dataset
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=VAL_SIZE, shuffle=True, random_state=SEED
        )

        X_train, X_test, y_train, y_test = train_test_split(
            X_train, y_train, test_size=TEST_SIZE, shuffle=True, random_state=SEED
        )

        print("Start training...")
        self.model.train(X_train, y_train)
        print("Training completed!")

        # evaluate
        y_val_pred = self.model.predict(X_val)
        y_test_pred = self.model.predict(X_test)

        val_acc = accuracy_score(y_val, y_val_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        print(f"Val accuracy: {val_acc}")
        print(f"Test accuracy: {test_acc}")

        return self.model, self.vectorizer, self.label_encoder


## **7. Thực hiện dự đoán**

In [65]:
class Predictor:

    def __init__(self, model, vectorizer, label_encoder):

        self.model = model
        self.vectorizer = vectorizer
        self.label_encoder = label_encoder
        self.preprocessor = TextPreprocessor()

    def predict(self, text: str):

        processed = self.preprocessor.preprocess(text)

        features = self.vectorizer.transform(processed)
        features = np.array(features).reshape(1, -1)

        pred = self.model.predict(features)

        label = self.label_encoder.inverse_transform(pred)[0]

        return label


In [67]:
test_input = 'I am actually thinking a way of doing something useful'

# Reload original messages from dataframe since they were preprocessed earlier
original_messages = df['Message'].values.tolist()
original_labels = df['Category'].values.tolist()

trainer = Trainer()
model, vectorizer, le = trainer.train(original_messages, original_labels)
predictor = Predictor(model, vectorizer, le)
prediction_cls = predictor.predict(test_input)
print(f'Prediction: {prediction_cls}')

Start training...
Training completed!
Val accuracy: 0.9820627802690582
Test accuracy: 0.9802867383512545
Prediction: ham
